In [2]:
import yfinance as yf
import pandas as pd
import pathlib
import numpy as np
import scipy.optimize as sci_opt

In [3]:
# Define the symbols
symbols = ['KO', 'WMT', 'VOO']

# If we don's have the data then grab it
if not pathlib.Path(r'C:\Users\carle\Documents\Portafolio Optimization\Yahoo\data\stock_data.csv').exists():

    # Inizialize the client 
    price_data_frame = yf.download(symbols, start='2020-01-01', end='2024-01-01')
    
    # Save the csv
    price_data_frame.to_csv(r'C:\Users\carle\Documents\Portafolio Optimization\Yahoo\data\stock_data.csv')
    print(price_data_frame)

else:

    # Load the data
    price_data_frame = pd.read_csv(r'C:\Users\carle\Documents\Portafolio Optimization\Yahoo\data\stock_data.csv')
    print(price_data_frame)


# The data is close to what we need but we need a single column for each symbol, where each column contains the close price
price_data_frame = pd.read_csv(
    r'C:\Users\carle\Documents\Portafolio Optimization\Yahoo\data\stock_data.csv',
    header=[0, 1],  
    index_col=0      
)

price_data_frame = price_data_frame['Close']


           Price               Close            Close.1             Close.2  \
0         Ticker                  KO                VOO                 WMT   
1           Date                 NaN                NaN                 NaN   
2     2020-01-02   45.43271255493164   271.902099609375   36.35920715332031   
3     2020-01-03  45.184852600097656  269.9157409667969   36.03823471069336   
4     2020-01-06  45.168331146240234  270.9271240234375   35.96487045288086   
...          ...                 ...                ...                 ...   
1003  2023-12-22  54.594417572021484  423.0841979980469  51.004302978515625   
1004  2023-12-26  54.819087982177734  424.8434753417969   50.92616271972656   
1005  2023-12-27  54.959495544433594  425.6210021972656   51.40479278564453   
1006  2023-12-28    54.9969482421875  425.6890563964844  51.303855895996094   
1007  2023-12-29   55.16545104980469  424.5518798828125   51.32990264892578   

                    High              High.1       

We have N assets. Weights and returns are vectors.

$$
w =
\begin{bmatrix}
w_1 \\
w_2 \\
\vdots \\
w_N
\end{bmatrix},
\quad
\mu =
\begin{bmatrix}
\mu_1 \\
\mu_2 \\
\vdots \\
\mu_N
\end{bmatrix}
$$

Covariance matrix:

For a portfolio with $N$ assets, let the weights vector be

$$
w =
\begin{bmatrix}
w_1 \\
w_2 \\
\vdots \\
w_N
\end{bmatrix}
\in \mathbb{R}^{N \times 1},
\qquad
\mu =
\begin{bmatrix}
\mu_1 \\
\mu_2 \\
\vdots \\
\mu_N
\end{bmatrix}
\in \mathbb{R}^{N \times 1}
$$

and let the covariance matrix be

$$
\Sigma =
\begin{bmatrix}
\mathrm{Var}(R_1) & \mathrm{Cov}(R_1,R_2) & \cdots & \mathrm{Cov}(R_1,R_N) \\
\mathrm{Cov}(R_2,R_1) & \mathrm{Var}(R_2) & \cdots & \mathrm{Cov}(R_2,R_N) \\
\vdots & \vdots & \ddots & \vdots \\
\mathrm{Cov}(R_N,R_1) & \mathrm{Cov}(R_N,R_2) & \cdots & \mathrm{Var}(R_N)
\end{bmatrix}
\in \mathbb{R}^{N \times N}
$$

Then the portfolio expected return is

$$
\mathbb{E}[R_p] = w^\top \mu
$$

and the portfolio variance is

$$
\mathrm{Var}(R_p) = w^\top \Sigma w
$$

This is a vector-matrix-vector product:
$$
(1 \times N)(N \times N)(N \times 1) = (1 \times 1)
$$
so the result is a scalar.

Finally, portfolio volatility is the square root of variance:

$$
\sigma_p = \sqrt{w^\top \Sigma w}
$$

With daily data, annualization is usually written as

$$
\mu_{\text{annual}} = 252\,\mu,
\qquad
\Sigma_{\text{annual}} = 252\,\Sigma
$$

In [4]:
log_return = np.log(1 + price_data_frame.pct_change())

# With a given set of weights, the function returns the portafolio expected retrun, volativility (desv st), and sharpe ratio.
def get_metrics(weights: list) -> np.array:

    # Convert to a numpy array
    weights = np.array(weights)

    # Calculate anualized return
    ret = np.sum(log_return.mean()*weights)*252

    # Calculate the volatibility 
    vol = np.sqrt(
        np.dot(weights.T, np.dot(log_return.cov() * 252, weights))
    )

    # Calculate the sharpe ratio, assuming 0 risk free
    sr = ret / vol

    return np.array([ret, vol, sr])



In [5]:
# We have to negativiza the sharpe ratio, in order to minimize
def grab_negative_sharpe(weights: list) -> np.array:
    return get_metrics(weights)[2]*(-1)

def grab_volatility(weights: list) -> np.array:
    return get_metrics(weights)[1]

# Scipy requires equality constraints (type='eq') to return 0
# We need the weights to sum to 1, so we return sum - 1
# When weights sum to 1 -> 1 - 1 = 0 -> constraint satisfied

# Restriction 
def check_sum(weights: list) -> float:
    return np.sum(weights) - 1

In [6]:
# Check the number of assets
num_of_symbols = len(symbols)

# Sequence of (min, max) pairs for each element in x. None is used to specify no bound - Used in sci_opt.minimize
bounds = tuple((0,1) for symbol in range(num_of_symbols))

# Restriction weights = 100
# Constraints for SLSQP are defined as a list of dictionaries.
constraints = ({'type': 'eq', 'fun': check_sum})

# Create initial guest of weights to start the minimization, let's start with the same fraction foreach asset
init_guess = num_of_symbols * [1 / num_of_symbols]

# Perform the operation to minimize risk 
optimized_sharpe = sci_opt.minimize(
    grab_negative_sharpe,   # Minimize this
    init_guess,
    method='SLSQP',
    bounds=bounds,  # Don's exced these bounds
    constraints=constraints
)

# Print the results.
print('')
print('='*80)
print('OPTIMIZED SHARPE RATIO:')
print('-'*80)
print(optimized_sharpe)
print('-'*80)





OPTIMIZED SHARPE RATIO:
--------------------------------------------------------------------------------
 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -0.5092192854084198
       x: [ 0.000e+00  6.979e-01  3.021e-01]
     nit: 5
     jac: [ 1.439e-01 -2.534e-04  5.821e-04]
    nfev: 21
    njev: 5
--------------------------------------------------------------------------------


## Portfolio Optimization

The goal is to find the optimal asset weights that maximize the **Sharpe Ratio** 
(return per unit of risk). Since most optimizers minimize, we flip the sign:

$$\min_{w} \left( -\frac{\mu_p}{\sigma_p} \right)$$

Where:

$$\mu_p = \sum_{i} w_i \mu_i \times 252 \quad \text{(annualized return)}$$

$$\sigma_p = \sqrt{w^T \Sigma w} \quad \text{(annualized volatility)}$$

Subject to:

$$\sum_{i} w_i = 1 \qquad 0 \leq w_i \leq 1$$

---

### Method: SLSQP

At each iteration, SLSQP solves a **quadratic subproblem** to find the best 
direction to move the weights, respecting the constraints at every step, until 
no further improvement is possible.

---

### Implementation
```python
sci_opt.minimize(
    grab_negative_sharpe,   # function to minimize: -Sharpe
    init_guess,             # starting point: equal weights [1/3, 1/3, 1/3]
    method='SLSQP',
    bounds=bounds,          # each weight: 0 ≤ w ≤ 1
    constraints=constraints # weights must sum to 1
)
```

The optimizer calls `grab_negative_sharpe` and `check_sum` internally on each 
iteration — you don't call them manually. It stops when no weight adjustment 
improves the Sharpe without violating the constraints.

In [7]:
# Define the boundaries for each symbol. Remember I can only invest up to 100% of my capital into a single asset.
bounds = tuple((0, 1) for symbol in range(num_of_symbols))

# Define the constraints, here I'm saying that the sum of each weight must not exceed 100%.
constraints = ({'type': 'eq', 'fun': check_sum})

# We need to create an initial guess to start with,
# and usually the best initial guess is just an
# even distribution. In this case 25% for each of the 4 stocks.
init_guess = num_of_symbols * [1 / num_of_symbols]

# Perform the operation to minimize the risk.
optimized_volatility = sci_opt.minimize(
    grab_volatility, # minimize this.
    init_guess, # Start with these values.
    method='SLSQP',
    bounds=bounds, # don't exceed these bounds.
    constraints=constraints # make sure you don't exceed the 100% constraint.
)

# Print the results.
print('')
print('='*80)
print('OPTIMIZED VOLATILITY RATIO:')
print('-'*80)
print(optimized_volatility)
print('-'*80)


OPTIMIZED VOLATILITY RATIO:
--------------------------------------------------------------------------------
 message: Optimization terminated successfully
 success: True
  status: 0
     fun: 0.18942419839507357
       x: [ 3.748e-01  2.378e-01  3.873e-01]
     nit: 7
     jac: [ 1.893e-01  1.892e-01  1.897e-01]
    nfev: 28
    njev: 7
--------------------------------------------------------------------------------


### Extra explanation why KO is zero

The three assets share a very similar volatility, so the main differentiator is return. The optimizer excluded KO because it delivers roughly half the return of VOO with the same level of risk, making it the worst return/risk ratio of the three.

In [13]:
# Expected retrun anualized 
expected_returns = log_return.mean()

# Anual volatility
volatility = log_return.std()*np.sqrt(252)

# Sharpe individual (risk free = 0)
sharpe = expected_returns / volatility

metrics = pd.DataFrame({
    "Return": expected_returns,
    "Volatility": volatility,
    "Sharpe": sharpe
})

print(metrics)

          Return  Volatility    Sharpe
Ticker                                
KO      0.000193    0.224857  0.000859
VOO     0.000443    0.230694  0.001922
WMT     0.000343    0.236520  0.001451


Looking at the correlation matrix, KO shows a relatively high correlation with VOO (0.68), meaning they tend to move together. This eliminates any meaningful diversification benefit that could have justified including KO despite its lower return.


In [14]:
correlation_matrix = log_return.corr()
print(correlation_matrix)

Ticker        KO       VOO       WMT
Ticker                              
KO      1.000000  0.683435  0.411263
VOO     0.683435  1.000000  0.467788
WMT     0.411263  0.467788  1.000000
